<a href="https://colab.research.google.com/github/blancavazquez/Programa-intensivo-2026-Modulos3y5/blob/main/notebooks/sesion_viernes_pyspark/Practica_Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica Spark Streaming con Wikimedia EventStreams

El objetivo de esta práctica es construir una aplicación que permita la **ingesta de datos en tiempo real** usando Apache Kafka y su procesamiento como flujos de datos estructurados mediante **PySpark Structured Streaming**.



![my_test_image](https://turing.iimas.unam.mx/~blancavg/images/app_twitter.png)

En esta libreta se utilizará el flujo público `recentchange` de Wikimedia EventStreams. Cada evento corresponde a un cambio reciente realizado en alguno de los proyectos públicos de Wikimedia y se recibe en formato JSON.

La arquitectura de la práctica es:

```text
Producer (Wikimedia EventStreams (SSE))
            ↓
          Kafka
            ↓
PySpark Structured Streaming (Spark 4.0.4)
```

Los datos se consumen desde el endpoint:

`https://stream.wikimedia.org/v2/stream/recentchange`

# ¿Qué es Apache Kafka?
Kafka es una plataforma distribuida de transmisión de datos que permite publicar, almacenar, y procesar en tiempo real. Kafka es un proyecto de código abierto desarrollado por LinkedIn y donado a la Apache Software Foundation escrito en Java y Scala.

Ventajas que ofrece Kafka en comparación a otras opciones para el manejo de datos más tradicionales:

* Garantiza un procesamiento rápido y fluido de datos.
* Escalabilidad
* Tolerancia a fallos
* Pueden manejar escritura y lectura de datos de alta frecuencia
* Unifica los canales de datos entre microservicios (contenedores).

# Conceptos

* Topico: Un tópico es un flujo de datos, los cuales se guardan en forma formato (llave, valor). Cada evento (dato) que llega al sistema debe ser parte de un tópico.

* Productores: Son las apps o microservicios que publican datos en el sistema de Kafka. Publican datos en los tópicos de su eleccion.

* Consumidores: Son las apps o microservicios que usan los datos publicados por los productores. Para consumir datos, un consumidor debe suscribirse a un tópico.

* Broker: es el conjunto de servidores (clúster) donde se ejecutar Kafka. Los datos de un tópico son replicados y particionados en varios brokers, los cual permite a los consumidores leer datos en paralelo y tolerar fallos.

# Instalación y configuración de Apache Kafka

In [ ]:
""" Celda 1
Spark 4.0.4 usa Scala 2.13. Para Structured Streaming con Kafka,
cargamos únicamente el conector spark-sql-kafka correspondiente.
"""

import os
os.environ['PYSPARK_SUBMIT_ARGS'] = ('--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.4 pyspark-shell')


In [ ]:
"""
Celda 2:
Dependencias del productor Wikimedia → Kafka
"""
!pip install -q kafka-python requests-sse

In [ ]:
"""
 Celda 3
Instalación de Kafka 3.8.0 (broker)
[toma alrededor de 10 minutos la descarga e instalación]
"""
!wget -q https://archive.apache.org/dist/kafka/3.8.0/kafka_2.12-3.8.0.tgz
!tar -xzf kafka_2.12-3.8.0.tgz


In [ ]:
!/content/kafka_2.12-3.8.0/bin/zookeeper-server-start.sh -daemon /content/kafka_2.12-3.8.0/config/zookeeper.properties
!/content/kafka_2.12-3.8.0/bin/kafka-server-start.sh -daemon /content/kafka_2.12-3.8.0/config/server.properties

!echo "Esperando 10 segundos mientras Kafka y ZooKeeper se inician..."
!sleep 10


In [ ]:
!ps -ef | grep kafka

In [ ]:
# Crear el tópico solo si todavía no existe.
! /content/kafka_2.12-3.8.0/bin/kafka-topics.sh --create --bootstrap-server 127.0.0.1:9092 --replication-factor 1 --partitions 1 --topic proyecto 2>/dev/null || true


# Productor: Wikimedia EventStreams → Kafka

Wikimedia publica los cambios recientes mediante **Server-Sent Events (SSE)**. El productor se conecta al stream y envía cada evento completo, en formato JSON, al tópico `proyecto` de Kafka.

> **Nota:** esta celda queda ejecutándose continuamente. Para detenerla, interrumpe la ejecución de la celda.

In [ ]:
import json
from kafka import KafkaProducer
from requests_sse import EventSource

URL = "https://stream.wikimedia.org/v2/stream/recentchange"

producer = KafkaProducer(bootstrap_servers="localhost:9092",value_serializer=lambda x: json.dumps(x).encode("utf-8"))

headers = {"User-Agent": "PySpark-Streaming-Workshop/1.0"}

print("Escuchando cambios recientes de Wikimedia en tiempo real...\n")

with EventSource(URL, headers=headers) as stream:
    for event in stream:
        if event.type != "message":
            continue

        try:
            change = json.loads(event.data)
        except ValueError:
            continue

        # Ignorar eventos de prueba generados por la infraestructura.
        if change.get("meta", {}).get("domain") == "canary":
            continue

        # Mostrar algunos campos del evento recibido.
        print(
            f"Wiki: {change.get('wiki')} | "
            f"Usuario: {change.get('user')} | "
            f"Página: {change.get('title')} | "
            f"Tipo: {change.get('type')}"
        )

        # Publicar el evento completo en Kafka.
        producer.send("proyecto", value=change)
        producer.flush()

# Consumiendo datos con PySpark Structured Streaming

### Importante: reiniciar el kernel

Si previamente se ejecutó la libreta con otra versión de Spark o del conector Kafka, **reinicia el kernel antes de ejecutar esta celda**. Los JAR de Spark/Kafka se cargan en la JVM al iniciar Spark y no deben mezclarse versiones dentro de la misma sesión.

La práctica está preparada para **Spark/PySpark 4.0.4 + Scala 2.13 + `spark-sql-kafka-0-10_2.13:4.0.4`**.


In [ ]:
!pip install kafka-python

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, LongType, BooleanType)
from pyspark.sql.functions import from_json, col, to_timestamp, window, desc

spark = (
    SparkSession.builder
    .appName("wikimedia-streaming")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")


In [ ]:
import pyspark
sc = spark.sparkContext
scala_version = sc._gateway.jvm.scala.util.Properties.versionNumberString()

print("Scala version:", scala_version)
print("Spark version:", sc.version)
print("PySpark version:", pyspark.__version__)


# Ejercicio 1: Análisis de actividad histórica

Pregunta: **¿qué proyectos de Wikimedia presentan mayor actividad durante cada intervalo de un minuto?**

Algunos proyectos que podemos encontrar son:

* enwiki → Wikipedia en inglés
* eswiki → Wikipedia en español
* commonswiki → Wikimedia Commons
* frwiki → Wikipedia en francés
* dewiki → Wikipedia en alemán

### Paso 1: Leer los eventos de kafka

El prámetro de **earliest** indica:
* Procesa los eventos que ya existen en Kafka;
* Continúa escuchando;
* Procesa también los nuevos eventos que lleguen.

In [ ]:
events_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "proyecto")
    .option("startingOffsets", "earliest") # IMPORTANTE
    .load()
)

El código anterior, nos entrega algo parecido a:

* key
* **value**
* topic
* partition
* offset
* timestamp
* timestampType

Pongamos atención a "value", la cual contiene el JSON:

{

  "type": "edit",

  "title": "Python",

  "timestamp": 1756900000,

  "user": "Usuario123",

  "bot": false,

  "wiki": "eswiki"

}

## Paso 2: Crear una ventana temporal

In [ ]:
events_by_wiki = (
    events_df
    .groupBy("partition")
    .count()
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("test_kafka")
    .start()
)

In [ ]:
spark.sql("SELECT * FROM test_kafka").show() #Esta salida nos confirma que sí hay eventos y Spark puede leerlos.

In [ ]:
#Comando para detener una consulta
events_by_wiki.stop()

## Paso 3: Parsear el JSON de wikipedia

In [ ]:

from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

#definimos el esquema para cada campo
wiki_schema = StructType([
    StructField("type", StringType(), True), #este campo puede ser NULL (Spark no considera un error que eventualmente un evento no tenga ese campo.)
    StructField("title", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("wiki", StringType(), True)])

parsed_events_df = (
    events_df
    .select(from_json(col("value").cast("string"),
                      wiki_schema).alias("data")) #alias data: creamos una estructura temporal
    .select("data.*") #Extrae todos los campos que están dentro de data
)

In [ ]:
#Parsear también el formato de tiempo
from pyspark.sql.functions import from_unixtime
parsed_events_df = (parsed_events_df.withColumn("event_time",from_unixtime(col("timestamp")).cast("timestamp")))

In [ ]:
#comprobando el esquema
parsed_events_df.printSchema()

## Paso 4: Pasamos de los eventos individuales de Wikimedia a una medida de actividad por proyecto y por minuto.


* Función window de PySpark, nos permite dividir los eventos del flujo en intervalos de tiempo.
* Parámetro: watermark ayuda a Spark a determinar cuándo puede considerar que una ventana temporal está suficientemente completa, teniendo en cuenta que algunos eventos pueden llegar retrasados.
* Creamos ventanas de tiempo:

10:15:00 ───────── 10:16:00
       Ventana 1

10:16:00 ───────── 10:17:00
       Ventana 2

10:17:00 ───────── 10:18:00
       Ventana 3

In [ ]:
"""
¿Cuántos cambios tuvo cada proyecto Wikimedia durante cada minuto?
"""

from pyspark.sql.functions import window

activity_by_wiki = (
    parsed_events_df
    .withWatermark("event_time", "2 minutes") # ¿cuánto retraso tolero? watermark permite a Spark manejar eventos que llegan retrasados.
                                              # Cierra ventanas antiguas y liberara memoria
                                              # ¿por qué deberíamos aplicar tolerancia al recibir datos?
    .groupBy(window(col("event_time"), "1 minute"), col("wiki"))#Quiero agrupar datos en intervarlos de 1 minuto
    .count() # contamos los eventos
)

In [ ]:
"""
Ejecución de la consulta
"""

activity_query = (
    activity_by_wiki
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("activity_by_wiki")
    .start()
)

In [ ]:
"""
Visualización
"""

spark.sql("""
    SELECT
        window.start,
        window.end,
        wiki,
        count
    FROM activity_by_wiki
    ORDER BY window.start DESC, count DESC
""").show(20, truncate=False)

### Preguntas para discutir

1. ¿Qué proyectos de Wikimedia reciben más cambios?
2. ¿La actividad es constante o cambia con el tiempo?

# Ejercicio 2: Páginas con mayor actividad

Identificar continuamente las páginas que reciben más cambios durante una ventana de **2 minutos**, actualizada cada **30 segundos**.

In [ ]:
# Identificar páginas con mayor actividad en ventanas de 2 minutos,
# actualizadas cada 30 segundos.
top_pages = (
    parsed_events_df
    .withWatermark("event_time", "3 minutes")
    .groupBy(
        window(col("event_time"), "2 minutes", "30 seconds"),
        col("wiki"),
        col("title")
    )
    .count()
    .orderBy(desc("count"))
)

In [ ]:
top_pages_query = (
    top_pages
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("pages_by_wiki")
    .start()
)

In [ ]:
"""
Visualización
"""

spark.sql("""
    SELECT
        window.start,
        window.end,
        wiki,
        count
    FROM pages_by_wiki
    ORDER BY window.start DESC, count DESC
""").show(20, truncate=False)